In [4]:
#최종
#Best Parameters: {'colsample_bytree': 0.9, 'gamma': 2, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 900, 'subsample': 0.9}
#1.051
#Train ACC : 0.6862
#Val ACC : 0.6857

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything(42)  # 무조건 42로 세팅!!

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
submission = pd.read_csv('sample_submission.csv')

###################################################
###################################################

#1. session ID별 order의 진행률 열 만들기
Max_order = train.groupby("session ID")["order"].transform("max")
train["order_progess_rate"] = train["order"]**2 / Max_order

#2. 국가코드 0,1로 구분
train['is_country_29?'] = (train['country'] == 29)
train.loc[train['country'].isin([43,44,45,46,47]),'country'] = 43


#3.train과 test 데이터를 합쳐 수치화하고 분리하기
com = pd.concat([train['page 2 (clothing model)'], test['page 2 (clothing model)']])
com, valuga = pd.factorize(com)
train['page 2 (clothing model)'] = com[:len(train)]
test['page 2 (clothing model)'] = com[len(train):]


#4. 서로 결측치 채우기
dic = dict(zip(train['page 2 (clothing model)'], train['page 1 (main category)']))
train['page 1 (main category)'] = train['page 1 (main category)'].fillna(train['page 2 (clothing model)'].map(dic))
mode_mapping = train.groupby('page 1 (main category)')['page 2 (clothing model)'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
train['page 2 (clothing model)'] = train['page 2 (clothing model)'].fillna(train['page 1 (main category)'].map(mode_mapping))


#5.그럼에도 결측치가 있다면 채우기
train['page 1 (main category)'] = train['page 1 (main category)'].fillna(train['page 1 (main category)'].mode().values[0])
train['page 2 (clothing model)'] = train['page 2 (clothing model)'].fillna(train['page 2 (clothing model)'].mode().values[0])


#6. 이상치 상한값으로 대체하기
Q_1 = train['price'].quantile(0.25)
Q_3 = train['price'].quantile(0.75)
Ubound =int( Q_3 + 1.5 * (Q_3 - Q_1))
train.loc[(train['price'] > Ubound), 'price'] = Ubound


#7. 평일 주말 나누기
train.loc[train['date'].isin([ "2024-04-01","2024-04-06", "2024-04-07", "2024-04-13", "2024-04-14", "2024-04-20", "2024-04-21", "2024-04-27", "2024-04-28", "2024-04-30"]),'date'] = 1
train.loc[train['date'] != 1,'date'] = 0
train['date'] = train['date'].astype(int)


# 훈련, 테스트 데이터 나누기
X = train.drop(columns=['click_id', 'session ID', 'Clicked', 'colour', 'country',])
y = train.Clicked

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X.columns)
display(X.head())


#예측에 사용할 모델을 가져와서 학습
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV

para_xgb =  {'colsample_bytree': [0.9], 'gamma': [2], 'learning_rate': [0.01], 'max_depth': [4], 'min_child_weight': [5], 'n_estimators': [900], 'subsample': [0.9]}
Xgb= XGBClassifier(scale_pos_weight=1.051)
Gxgb = GridSearchCV(Xgb, param_grid=para_xgb, cv=3, scoring='accuracy',n_jobs=-1)
Gxgb.fit(X_train, y_train)


# 학습한 모델을 평가
from sklearn.metrics import accuracy_score

train_pred = Gxgb.predict(X_train)
val_pred = Gxgb.predict(X_val)

print("Best Parameters:", Gxgb.best_params_)
print("Train ACC : %.4f" % accuracy_score(y_train, train_pred))
print("Val ACC : %.4f" % accuracy_score(y_val, val_pred))


###################################################
###################################################
Max_order = test.groupby("session ID")["order"].transform("max")
test["order_progess_rate"] = test["order"]**2 / Max_order


test['is_country_29?'] = (test['country'] == 29)
test.loc[test['country'].isin([43,44,45,46,47]),'country'] = 43


dic = dict(zip(test['page 2 (clothing model)'], test['page 1 (main category)']))
test['page 1 (main category)'] = test['page 1 (main category)'].fillna(test['page 2 (clothing model)'].map(dic))
mode_mapping = test.groupby('page 1 (main category)')['page 2 (clothing model)'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
test['page 2 (clothing model)'] = test['page 2 (clothing model)'].fillna(test['page 1 (main category)'].map(mode_mapping))


test['page 1 (main category)'] = test['page 1 (main category)'].fillna(test['page 1 (main category)'].mode().values[0])
test['page 2 (clothing model)'] = test['page 2 (clothing model)'].fillna(test['page 2 (clothing model)'].mode().values[0])


Q_1 = test['price'].quantile(0.25)
Q_3 = test['price'].quantile(0.75)
Ubound =int( Q_3 + 1.5 * (Q_3 - Q_1))
test.loc[(test['price'] > Ubound), 'price'] = Ubound


test.loc[test['date'].isin([ "2024-04-01","2024-04-06", "2024-04-07", "2024-04-13", "2024-04-14", "2024-04-20", "2024-04-21", "2024-04-27", "2024-04-28", "2024-04-30"]),'date'] = 1
test.loc[test['date'] != 1,'date'] = 0
test['date'] = test['date'].astype(int)


test = test.drop(columns=['click_id', 'session ID','colour','country'])


test = test.reindex(columns=X.columns, fill_value=0)
###################################################
###################################################


# X_test를 만들어서 아래 코드를 실행하세요.
X_test = test
output = Gxgb.predict(X_test)
assert len(output) == 7695  # sanity check


# submission DataFrame을 만들어서 아래 코드를 실행하세요.
submission = pd.read_csv('sample_submission.csv')
submission['Clicked'] = output
submission.to_csv('submission.csv', index=False)

#최종
#Best Parameters: {'colsample_bytree': 0.9, 'gamma': 2, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 900, 'subsample': 0.9}
#1.051
#Train ACC : 0.6862
#Val ACC : 0.6857

Index(['date', 'order', 'page 1 (main category)', 'page 2 (clothing model)',
       'location', 'model photography', 'page', 'price', 'order_progess_rate',
       'is_country_29?'],
      dtype='object')


,date,order,page 1 (main category),page 2 (clothing model),location,model photography,page,price,order_progess_rate,is_country_29?
0,1,1,1.0,0,5,1,1,39,0.111111,True
1,1,2,1.0,1,6,1,1,40,0.444444,True
2,1,3,2.0,2,2,1,1,46,1.000000,True
3,1,4,2.0,3,6,2,1,30,1.777778,True
4,1,5,2.0,4,3,2,1,41,2.777778,True


Best Parameters: {'colsample_bytree': 0.9, 'gamma': 2, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_weight': 5, 'n_estimators': 900, 'subsample': 0.9}
Train ACC : 0.6862
Val ACC : 0.6857


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
